## Block 1 — Environment Initialization and Spark Session Configuration
**Rationale:** Establish the distributed execution environment and configure necessary dependencies for LLM-based feature extraction.

**Technical Implementation:** Initialize a local Spark session with customized timeout configurations to prevent worker nodes from dropping due to high-latency asynchronous API calls.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from google.colab import drive

# Ensure worker nodes have required libraries for text processing
!pip install -q langchain-openai langgraph langchain

# Establish connection to persistent storage
drive.mount('/content/drive')

# Align Python runtime environments across driver and workers to prevent serialization mismatch
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Initialize Spark with relaxed timeouts to accommodate synchronous LLM API limits
spark = (
    SparkSession.builder
    .appName("BookFeatureExtraction")
    .master("local[*]")
    .config("spark.python.worker.timeout", "1200")
    .config("spark.rpc.askTimeout", "1200s")
    .config("spark.network.timeout", "1200s")
    .config("spark.executor.heartbeatInterval", "1100s")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.7 MB/s eta 0:00:00
Mounted at /content/drive


## Block 2 — Data Ingestion
**Rationale:** Ingest raw review data from distributed storage into a structured dataframe for downstream processing.

**Technical Implementation:** Utilize Spark's lazy evaluation to read CSV data with inferred schemas, executing a lightweight `.show()` action strictly for schema validation and debugging.

In [ ]:
raw_data_path = "/content/drive/MyDrive/BT4221/split_review_4" # Change number accordingly

# Read metadata without eagerly loading the entire dataset into memory
raw_reviews_df = spark.read.csv(raw_data_path, header=True)
print(f"Successfully loaded data from: {raw_data_path}")

# Materialize a minimal subset to verify ingestion integrity
raw_reviews_df.show(5, truncate=False)

Successfully loaded data from: /content/drive/MyDrive/BT4221/split_review_4
+---------------------+----------+-------+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Block 3 — LLM-Driven Feature Extraction via MapPartitions
**Rationale:** Extract structured metadata (genres, tropes) from unstructured text using Large Language Models to enrich the dataset for advanced analytics.
**Technical Implementation:** Apply a custom zero-closure function using `mapPartitions` to minimize serialization overhead between the JVM and Python processes. Implement asynchronous LangGraph workflows within workers to parallelize OpenAI API requests effectively, followed by a `.cache()` to persist the expensive computation in memory.

In [ ]:
from pyspark.sql.types import StringType, ArrayType, StructField

# Define output schema prior to partition mapping
output_schema = raw_reviews_df.schema \
    .add(StructField("clean_trope", StringType(), True)) \
    .add(StructField("genres", ArrayType(StringType()), True))

def process_partition(iterator):
    # Imports localized to worker nodes to ensure availability during distributed execution
    from typing import List, TypedDict, Optional
    from pydantic import BaseModel, Field
    from langgraph.graph import StateGraph, END
    from langchain_openai import ChatOpenAI
    import asyncio
    from itertools import islice
    import time

    GENRES = [
        "Fantasy", "Sci-Fi", "Romance", "Thriller", "Mystery", "Horror",
        "Historical Fiction", "Contemporary", "Dystopian", "Memoir",
        "Biography", "Self-Help", "True Crime", "Graphic Novel", "Young Adult",
        "New Adult", "Western", "Literary Fiction", "Poetry", "Adventure"
    ]

    class BookState(BaseModel):
        genre: List[str] = Field(description="List of applicable genres from the allowed list.")
        trope: str = Field(description="The overarching plot trope. Give it in 1 short sentence.")

    class AgentState(TypedDict):
        raw_review: str
        book_features: Optional[BookState]
        is_valid: bool

    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    async def validate_input(state: AgentState):
        prompt = f"Is this text a book review? Answer only YES or NO: {state['raw_review']}"
        try:
            response = await llm.ainvoke(prompt)
            is_valid = "YES" in response.content.upper()
        except:
            is_valid = False
        return {"is_valid": is_valid}

    async def extract_book_features(state: AgentState):
        if not state.get("is_valid"):
            return {"book_features": None}

        structured_llm = llm.with_structured_output(BookState)
        system_msg = f"Extract genres and tropes. Allowed genres: {', '.join(GENRES)}"
        try:
            features = await structured_llm.ainvoke([
                ("system", system_msg), ("human", state["raw_review"])
            ])
            return {"book_features": features}
        except:
            return {"book_features": None}

    # Construct the state graph for the extraction workflow
    workflow = StateGraph(AgentState)
    workflow.add_node("validator", validate_input)
    workflow.add_node("extractor", extract_book_features)
    workflow.set_entry_point("validator")
    workflow.add_edge("validator", "extractor")
    workflow.add_edge("extractor", END)

    app = workflow.compile()

    def batch_iterator(it, size):
        while True:
            batch = list(islice(it, size))
            if not batch: break
            yield batch

    async def process_batch(batch_rows):
        tasks = [app.ainvoke({"raw_review": r["reviewText"] or ""}) for r in batch_rows]
        return await asyncio.gather(*tasks)

    # Process records in asynchronous batches to manage API rate limits
    for batch in batch_iterator(iterator, size=10):
        loop = asyncio.new_event_loop()
        results = loop.run_until_complete(process_batch(batch))
        loop.close()

        for row, res in zip(batch, results):
            if isinstance(res, Exception):
                yield (*row, f"Error: {str(res)}", [])
            elif res.get("is_valid") and res.get("book_features"):
                feat = res["book_features"]
                yield (*row, str(feat.trope), list(feat.genre))
            else:
                yield (*row, "Invalid/Failed", [])

        # Introduce synthetic delay to respect provider token buckets
        time.sleep(5)

total_partitions = 4

# Execute MapPartitions and enforce schema on resulting RDD
extracted_features_df = raw_reviews_df.repartition(total_partitions) \
                 .rdd \
                 .mapPartitions(process_partition) \
                 .toDF(output_schema)

# Materialize and cache the costly LLM transformations to avoid re-computation in subsequent actions
extracted_features_df.cache()
extracted_features_df.show(5, truncate=False)

Processing 14612 rows...
+---------------------+----------+-------+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Block 5 — Data Serialization and Export
**Rationale:** Persist the enriched dataset to durable storage for downstream modeling or integration.

**Technical Implementation:** Coalesce the distributed partitions and flatten array types using concat_ws to ensure compatibility with the flat CSV format prior to executing the terminal write action.

In [ ]:
import pyspark.sql.functions as F

output_path = "/content/drive/MyDrive/BT4221/output_features_4.csv" # Change number accordingly

# Flatten array structures to ensure CSV compatibility
export_df = extracted_features_df.withColumn(
    "genres", F.concat_ws(", ", F.col("genres"))
)

# Execute final DAG action and persist to disk
export_df.write.mode("overwrite").option("header", "true").csv(output_path)
print(f"DataFrame successfully exported to: {output_path}")